Grupo 4 - 13/05/2026 - IA - Trabajo IA - Motor de Búsqueda VP-Tree

# Notebook 4: Motor de Búsqueda con VP-Tree y Demo

Construcción del índice de búsqueda facial mediante VP-Tree, pipeline completo de consulta y demostración interactiva con Gradio desplegada en Hugging Face Spaces.

## 0. Instalación

In [1]:
%%capture
!pip install vptree deepface tf-keras gradio

In [2]:
import pickle
import time
import numpy as np
import vptree
from pathlib import Path
from deepface import DeepFace
import random
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

26-05-20 14:44:33 - Directory /root/.deepface has been created
26-05-20 14:44:33 - Directory /root/.deepface/weights has been created


## 1. Carga del Dataset de Caras y embeddings

In [3]:
!wget -q EMBEDDINGS_URL

with open('embeddings.pkl', 'rb') as f:
    embeddings = pickle.load(f)

print(f"[INFO] Embeddings cargados : {len(embeddings)}")
print(f"[INFO] Dimensión : {len(next(iter(embeddings.values())))}")

[INFO] Embeddings cargados : 38299
[INFO] Dimensión : 128


In [4]:
%%capture
!wget FACE_CROPS_URL -O carasRecortadas.zip
!unzip carasRecortadas.zip -d carasRecortadas

## 2. Construcción del VP-Tree

Construimos el VP-Tree siguiendo el articulo [PyImageSearch](https://pyimagesearch.com/2019/08/26/building-an-image-hashing-search-engine-with-vp-trees-and-opencv/). Lo unico que cambiamos es que vamos a usar  ladistancia euclídea entre embeddings de 128d.

In [24]:
paths = list(embeddings.keys())
vecs_arr= np.array([embeddings[k] for k in paths], dtype=np.float32)

In [25]:
items = list(zip(paths, [list(v) for v in vecs_arr]))

In [27]:
indices = list(range(len(paths)))

def distancia(a, b):
    va = vecs_arr[int(a)] if isinstance(a, (int, np.integer)) else np.array(a, dtype=np.float32)
    vb = vecs_arr[int(b)] if isinstance(b, (int, np.integer)) else np.array(b, dtype=np.float32)
    return float(np.linalg.norm(va - vb))

print("Building VPTree...")
tree = vptree.VPTree(indices, distancia)
print("VP-Tree construido")

Building VPTree...
VP-Tree construido


## 3. Pipeline de búsqueda

La función `buscar` recibe la ruta de una imagen  y devuelve los `top_k` recortes más similares del dataset. Internamente sigue el mismo flujo que el Notebook 2: extrae el embedding con Facenet y consulta el VP-Tree con `get_n_nearest_neighbors`.

In [28]:
def buscar(img_path, top_k=5):
    emb = DeepFace.represent(img_path=str(img_path), model_name='Facenet', enforce_detection=False)[0]['embedding']
    query_vec = np.array(emb, dtype=np.float32)
    results = tree.get_n_nearest_neighbors(query_vec, top_k)
    return [(float(d), paths[int(idx)]) for d, idx in results]

### 3.1 Prueba de búsqueda

Verificamos que todo funciona correctamente al ejecutarlo sobre una muestra de 5 caras del dataset.

In [29]:
carasPath = Path('carasRecortadas')
caras = list(carasPath.rglob('*.jpg'))
print(f"Caras disponibles localmente: {len(caras)}")

Caras disponibles localmente: 38299


Vamos a probar que el motor de busqueda esta funcionando correctamente con una muestra dfe 5 caras aleatorias.

In [30]:
random.seed(42)
consultas = random.sample(caras, 5)

In [31]:
for idx, query_path in enumerate(consultas):
    results = buscar(query_path, top_k=5)

    fig, axes = plt.subplots(1, len(results) + 1, figsize=(3 * (len(results) + 1), 3))

    axes[0].imshow(mpimg.imread(query_path))
    axes[0].set_title('Consulta', fontsize=9)
    axes[0].axis('off')

    for ax, (dist, nombre) in zip(axes[1:], results):
        ruta = carasPath / nombre
        if ruta.exists():
            ax.imshow(mpimg.imread(ruta))
        ax.set_title(f'd={dist:.2f}', fontsize=9)
        ax.axis('off')

    plt.suptitle(f'Prueba {idx + 1} — {query_path.name}', fontsize=11)
    plt.tight_layout()
    plt.show()

<Figure size 1800x300 with 6 Axes>

<Figure size 1800x300 with 6 Axes>

<Figure size 1800x300 with 6 Axes>

<Figure size 1800x300 with 6 Axes>

<Figure size 1800x300 with 6 Axes>

## 4. Demo con Gradio

Desplegamos el  motor de busqueda faciales en  HuggingFace Spaces. A diferencia de de los